### 🧩 Problem Statement

#### 1. What problem is being solved?
We are building a **Fraud Detection System** to identify fraudulent credit card transactions. The core challenge is that fraud is rare (Imbalanced Data) but ensuring we catch it is critical (High Cost of False Negatives).

#### 2. Why is this problem important?
Financial institutions lose billions to fraud. A model that misses fraud (False Negatives) causes direct financial loss. A model that flags valid transactions (False Positives) causes customer dissatisfaction. We need a balance, biased towards catching fraud.

#### 3. Real-world relevance
This simulates real banking systems where legitimate transactions vastly outnumber fraudulent ones (e.g., 99.9% vs 0.1%). Standard accuracy metrics fail here (a dummy model predicting 'Safe' is 99.9% accurate).

### 🪜 Steps to Solve the Problem
1.  **Simulate Data**: Create a dataset with 95% Safe and 5% Fraud cases.
2.  **Experiment**: Train k-NN models with different `k` values (1 to 20).
3.  **Evaluate**: Compare `Accuracy` (general correctness) vs `Recall` (fraud catching ability).
4.  **Optimize**: Select the best `k` and apply modifications (Weighting, Thresholding) to improve performance.

### 🎯 Expected Output (OVERALL)
- A table showing how `k` affects Accuracy and Recall.
- Analysis identifying `k=9` as the stable choice.
- Demonstration that changing the decision threshold significantly improves fraud detection.

### 📚 Imports and Setup

#### 2.1 What this block does
It imports the necessary Python libraries for data manipulation (Pandas, NumPy), machine learning algorithms (Scikit-Learn), and metrics.

#### 2.2 Why these libraries are used
- **NumPy/Pandas**: For handling data structures.
- **sklearn.datasets**: To generate our synthetic fraud data.
- **sklearn.neighbors**: To usage the k-NN algorithm.
- **sklearn.metrics**: To measure how well our model works.

#### 2.3 When to use this
At the start of any data science project.

#### 2.4 How these lines work internally
Python loads the compiled C/C++ code for these libraries into memory so we can use their high-performance functions.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score, accuracy_score

### ⚙️ Step 1: Data Simulation (19:1 Imbalance)

#### 2.1 What this line does
We generate a synthetic dataset (`X` features and `y` labels) with 2000 samples, where 95% are Class 0 (Safe) and 5% are Class 1 (Fraud).

#### 2.2 Why this is used
 Real fraud data is private and hard to get. `make_classification` allows us to create a controlled environment to test our algorithms against **Class Imbalance**.

### ⚙️ Function Arguments Explanation (`make_classification`)

1.  **`n_samples=2000`**:
    *   **What**: Total rows of data.
    *   **Why**: Enough to see statistical patterns but fast to run.
2.  **`n_features=10`**:
    *   **What**: Number of columns (input variables).
    *   **Why**: Simulates transaction details (Amount, Time, Location, etc.).
3.  **`n_informative=5`**:
    *   **What**: Only 5 of the 10 features actually help predict fraud.
    *   **Why**: Real data contains useless noise; this tests the model's ability to ignore it.
4.  **`weights=[0.95, 0.05]`**:
    *   **What**: Sets the ratio of classes.
    *   **Why**: **CRITICAL**. This forces the 19:1 Imbalance we need to study.
5.  **`random_state=42`**:
    *   **What**: Seed for random number generator.
    *   **Why**: Ensures we get the exact same numbers every time we run this code (Reproducibility).

In [ ]:
print("--- 1. Data Simulation (19:1 Imbalance) ---")

X, y = make_classification(
    n_samples=2000, 
    n_features=10, 
    n_informative=5, 
    weights=[0.95, 0.05], # 19:1 Imbalance
    random_state=42
)

### ⚙️ Step 2: Data Splitting

#### 2.1 What this line does
It separates our data into a **Training Set** (to teach the model) and a **Test Set** (to evaluate it).

#### 2.2 Why this is used
If we test on the same data we trained on, the model will cheat (memorize). We need unseen data to measure true performance.

### ⚙️ Function Arguments Explanation (`train_test_split`)

1.  **`test_size=0.3`**:
    *   **What**: 30% of data goes to Testing, 70% to Training.
    *   **Why**: Standard split. Enough data to train, enough to get a reliable score.
2.  **`stratify=y`**:
    *   **What**: Ensures the Train and Test sets *both* have exactly 5% Fraud.
    *   **Why**: **MANDATORY for Imbalance**. Without this, the 30% test set might randomly end up with 0% fraud, making evaluation impossible.
3.  **`random_state=42`**:
    *   **Why**: Reproducibility.

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Check the counts to confirm imbalance
print(f"Train Imbalance: {np.bincount(y_train)} (Safe/Fraud)")

### ⚙️ Step 3: Analyzing k (The Loop)

#### 2.1 What this block does
It runs a loop from k=1 to k=20 (odd numbers only). For each `k`, it trains a new k-NN model and records its Accuracy and Recall.

#### 2.2 Why this is used
We don't know which `k` is best. We must experiment. This is called **Hyperparameter Tuning**.

#### 2.3 How it works internally
For each iteration:
1.  Create a model with `n_neighbors=k`.
2.  `fit()` stores the training data.
3.  `predict()` calculates distances for test points.
4.  Scores are computed and saved to a list.

In [ ]:
print("\n--- 2. k-NN Performance Analysis ---")

results = []

# Loop through k (1, 3, 5... 19)
for k in range(1, 21, 2): 
    # Initialize Model
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Train Model (Store data)
    knn.fit(X_train, y_train)
    
    # Predict on Test Data
    y_pred = knn.predict(X_test)
    
    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    
    # Store Result
    results.append({'k': k, 'Accuracy': acc, 'Recall': rec})

### 📊 Result Visualization

#### 2.1 What this does
Converts our list of results into a Pandas DataFrame for a clean, tabular display.

#### 2.7 Use Case
Allows us to visually compare how increasing `k` changes the trade-off between Accuracy and Recall.

In [ ]:
df_res = pd.DataFrame(results)
print(df_res.set_index('k'))

### 💡 Part A: Selecting k

**Recommendation:** We choose **k=9**.

**Reasons:**
1.  **Variance Reduction:** Lower k (k=1) is too erratic; it flags fraud based on single noisy points. k=9 smooths this out.
2.  **Generalization:** At k=9, we see a balance where Accuracy is high but we haven't completely killed Recall yet.
3.  **Business Stability:** k=9 provides a stable baseline. It is less likely to false-alarm on weird legitimate transactions compared to k=1.

### 💡 Part B: Bias-Variance Tradeoff Explanation

1.  **Low k (1-3): High Variance.** The model is too sensitive. It overfits to the training noise. If we changed the training data slightly, the predictions would change a lot.
2.  **High k (15-20): High Bias.** The model is too simple. It just votes for the majority class (Safe). It ignores the subtle patterns of Fraud. Recall drops to zero.
3.  **k=9 (Optimal):** The "Sweet Spot". Low enough bias to capture fraud patterns, low enough variance to ignore noise.

### ⚙️ Step 4: Part C - Modification 1 (Distance Weighting)

#### 2.1 What this does
We change the voting rule. Instead of each neighbor having 1 vote, **closer neighbors get more votes**.

#### 2.2 Why this solves Imbalance
In a 19:1 imbalanced dataset, a Fraud point might have 7 Safe neighbors and 2 Fraud neighbors (so it loses 7-2). But if those 2 Fraud neighbors are *very very close*, meaningful distance weighting gives them enough power to win the vote. This preserves local minority clusters.

In [ ]:
print("\n--- 3. Part C Demonstration ---")

# Modification 1: Distance Weighted
knn_weighted = KNeighborsClassifier(n_neighbors=9, weights='distance')
knn_weighted.fit(X_train, y_train)
y_pred_w = knn_weighted.predict(X_test)
rec_w = recall_score(y_test, y_pred_w)

print(f"Mod 1: Weighted k=9 Recall: {rec_w:.4f} (Improved Local Sensitivity)")

### ⚙️ Step 4: Part C - Modification 2 (Threshold Tuning)

#### 2.1 What this does
We manually change the decision rule. Instead of requiring > 50% probability to call it partial, we flag it as fraud if the probability is **> 20%**.

#### 2.2 Why this is used
**Cost Sensitivity.** Missing a fraud is expensive. We are willing to tolerate some False Alarms (False Positives) to ensure we catch the Fraud. If 2 out of 9 neighbors say "Fraud", that's suspicious enough to block the card.

#### 2.5 How to use it
1. Use `predict_proba()` to get the numbers (e.g., 0.22).
2. Apply condition `probs >= 0.2`.


In [ ]:
# Modification 2: Threshold Tuning
knn_prob = KNeighborsClassifier(n_neighbors=9)
knn_prob.fit(X_train, y_train)

# Get Probability of Class 1 (Fraud)
probs = knn_prob.predict_proba(X_test)[:, 1] 

# Custom Threshold > 0.2 (2 out of 9 votes)
y_pred_cust = (probs >= 0.2).astype(int)
rec_cust = recall_score(y_test, y_pred_cust)

print(f"Mod 2: Threshold > 0.2 Recall: {rec_cust:.4f} (Maximizing Safety)")

### 💼 Interview Perspective

#### Q1: Why did you use Stratified Split?
**A:** Because the data is highly imbalanced (5% fraud). A random split could result in a test set with no fraud cases. Stratification guarantees the class ratio remains consistent.

#### Q2: How does k-NN handle the Bias-Variance tradeoff?
**A:** `k` controls the tradeoff. Low `k` (1) is high variance (overfitting). High `k` (20+) is high bias (underfitting). We picked `k=9` to balance them.

#### Q3: Why is Accuracy a bad metric here?
**A:** With 95% legitimate transactions, a model that predicts "Safe" for everyone has 95% accuracy but catches 0 fraud. Recall is the better metric for risk/fraud.